# Result - Shared Topics 해석 만들기
- /home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv 문서와 /home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_industry_lda_bow.csv 문서에 있는 특정 토픽의 대표 문서 5건 정도를 내가 볼 수 있으면 좋을 거 같아. 서로 어떤 내용을 담고 있는지에 대해 파악할 수 있게.
- 해당 문서에는 Korean_summary랑 abstract이 있으니까 같이 출력해서 대표문서가 각각 어떤 내용을 담고 있는지 파악할 수 있게 만들어줘.
- 이왕이면 문서로 저장하기보다 노트북 환경에서 바로 확인할 수 있으면 좋을 거같아.

In [21]:
import pandas as pd

# 1. 데이터 로드
academia_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv"
industry_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_industry_lda_bow.csv"

df_academia = pd.read_csv(academia_path)
df_industry = pd.read_csv(industry_path)

print(f"Academia: {len(df_academia)}건 / Industry: {len(df_industry)}건")
print(f"Academia 토픽 범위: {df_academia['topic'].min()} ~ {df_academia['topic'].max()}")
print(f"Industry 토픽 범위: {df_industry['topic'].min()} ~ {df_industry['topic'].max()}")

# ============================================================
# 2. 함수 정의 (한 번만 실행)
# ============================================================

def show_topic_docs(df, topic_num, corpus_name="", n=5):
    """특정 토픽의 대표 문서 출력"""

    # 해당 토픽 문서만 필터링
    topic_df = df[df['topic'] == topic_num].copy()

    if len(topic_df) == 0:
        print(f"⚠️ Topic {topic_num}에 해당하는 문서가 없습니다.")
        return

    # topic_probability 기준 상위 n개
    top_docs = topic_df.nlargest(n, 'topic_probability')

    for i, (_, row) in enumerate(top_docs.iterrows(), 1):
        prob = row['topic_probability']

        # 제목 찾기
        title = row.get('title')

        # 연도 찾기
        year = row.get('date')
        if year and pd.notna(year):
            year = f" ({int(year)})"
        else:
            year = ""

        # 한국어 요약 찾기
        kor_sum = None
        for col in ['korean_summary']:
            if col in row.index and pd.notna(row.get(col)):
                kor_sum = str(row[col])[:500]
                break

        # Abstract 찾기
        abstract = None
        for col in ['abstract']:
            if col in row.index and pd.notna(row.get(col)):
                abstract = str(row[col])[:500]
                break

        print(f"\n{'─'*80}")
        print(f"📄 [{i}] 토픽확률: {prob:.4f}{year}")
        print(f"   📌 {title}")

        if kor_sum:
            print(f"\n   🇰🇷 한국어 요약:")
            print(f"      {kor_sum}{'...' if len(str(kor_sum)) >= 500 else ''}")

        if abstract:
            print(f"\n   📝 Abstract:")
            print(f"      {abstract}{'...' if len(str(abstract)) >= 500 else ''}")


def compare_topics(acad_topic, ind_topic, n=5):
    """Academia와 Industry 토픽 비교"""
    print("\n" + "🔬")
    print(f"  Academia Topic {acad_topic}  vs  Industry Topic {ind_topic}")
    print("🔬")

    show_topic_docs(df_academia, acad_topic, "ACADEMIA", n)
    print("\n" + "⬇️  ")
    show_topic_docs(df_industry, ind_topic, "INDUSTRY", n)


def topic_summary(df, corpus_name=""):
    """각 토픽별 문서 수 요약"""
    summary = df.groupby('topic').agg({
        'topic_probability': ['count', 'mean', 'max']
    }).round(4)
    summary.columns = ['문서수', '평균확률', '최대확률']
    print(f"\n📊 {corpus_name} 토픽별 요약:")
    print(summary)
    return summary


# ============================================================
# 3. 사용법 (아래 셀에서 실행)
# ============================================================
"""
# 토픽별 문서 수 확인
topic_summary(df_academia, "Academia")
topic_summary(df_industry, "Industry")

# 단일 코퍼스 특정 토픽 확인
show_topic_docs(df_academia, 3, "Academia", n=5)   # topic 3
show_topic_docs(df_industry, 5, "Industry", n=5)   # topic 5

# 두 토픽 비교 (Shared Topics 비교용)
compare_topics(3, 5, n=5)  # Academia topic 3 vs Industry topic 5
"""

# ============================================================
# 4. 여기서 토픽 번호 수정 후 실행!
# ============================================================

# 👇 확인하고 싶은 토픽 번호를 여기서 수정하세요
ACADEMIA_TOPIC = 11   # Academia 토픽 번호
INDUSTRY_TOPIC = 13   # Industry 토픽 번호
NUM_DOCS = 5         # 확인할 문서 수

# 실행 (주석 해제 후 사용)
compare_topics(ACADEMIA_TOPIC, INDUSTRY_TOPIC, NUM_DOCS)

Academia: 4800건 / Industry: 9665건
Academia 토픽 범위: 0 ~ 11
Industry 토픽 범위: 0 ~ 15

🔬
  Academia Topic 11  vs  Industry Topic 13
🔬

────────────────────────────────────────────────────────────────────────────────
📄 [1] 토픽확률: 0.6684 (2024)
   📌 Making Team Projects with Novices More Effective: An Experience Report

   🇰🇷 한국어 요약:
      이 연구는 컴퓨터 과학 캡스톤 프로젝트에서 이해관계자 참여의 통합을 탐구하며, 학생과 이해관계자 간의 협력 경험에 중점을 둡니다. 하와이 대학교 마노아 캠퍼스의 ICS 496 과정에서의 경험 보고서를 제시하며, 양측의 요구를 충족하는 소프트웨어 개발 프로세스를 구현하는 데 있어 직면한 도전과제를 강조합니다. 연구 결과는 수업 설계 수정이 중요하지만, 이러한 요구 사항이 실제 시나리오에서 어떻게 적용되는지에 대한 깊은 이해가 효과적인 협력을 위해 필요하다는 것을 시사합니다.

   📝 Abstract:
      Computer Science Capstone projects have shifted to offer more opportunities for students to engage with stakeholders as a team to simulate a real-life work scenario. Most works focus on the class design modifications and requirements for both students and stakeholders. However, discussions on how said requirements are instantiated in a concrete software development process th

## HTML

In [16]:
# ============================================================
# 🎯 토픽 대표 문서 뷰어 (HTML 버전) - Jupyter Notebook용
# ============================================================

import pandas as pd
from IPython.display import display, HTML

# 1. 데이터 로드
academia_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv"
industry_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_industry_lda_bow.csv"

df_academia = pd.read_csv(academia_path)
df_industry = pd.read_csv(industry_path)

print(f"Academia: {len(df_academia)}건 / Industry: {len(df_industry)}건")

# ============================================================
# 2. HTML 스타일 정의
# ============================================================

STYLE = """
<style>
.topic-container {
    font-family: 'Segoe UI', Arial, sans-serif;
    max-width: 100%;
    margin: 20px 0;
}
.topic-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    padding: 15px 20px;
    border-radius: 10px 10px 0 0;
    font-size: 18px;
    font-weight: bold;
}
.doc-card {
    border: 1px solid #e0e0e0;
    border-radius: 8px;
    margin: 10px 0;
    background: white;
    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}
.doc-header {
    background: #f8f9fa;
    padding: 12px 15px;
    border-bottom: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-between;
    align-items: center;
}
.doc-title {
    font-weight: bold;
    color: #2c3e50;
    font-size: 14px;
}
.doc-meta {
    color: #7f8c8d;
    font-size: 12px;
}
.prob-badge {
    background: #3498db;
    color: white;
    padding: 4px 10px;
    border-radius: 15px;
    font-size: 12px;
    font-weight: bold;
}
.doc-body {
    padding: 15px;
}
.section-label {
    font-weight: bold;
    color: #8e44ad;
    font-size: 12px;
    margin-bottom: 5px;
    display: flex;
    align-items: center;
    gap: 5px;
}
.section-content {
    color: #34495e;
    font-size: 13px;
    line-height: 1.6;
    margin-bottom: 12px;
    padding: 10px;
    background: #fafafa;
    border-radius: 5px;
    border-left: 3px solid #9b59b6;
}
.abstract-content {
    border-left-color: #3498db;
}
.divider {
    text-align: center;
    padding: 20px;
    font-size: 24px;
    color: #95a5a6;
}
.comparison-wrapper {
    display: flex;
    gap: 20px;
}
.corpus-column {
    flex: 1;
    min-width: 0;
}
</style>
"""

# ============================================================
# 3. HTML 생성 함수
# ============================================================

def get_topic_docs_html(df, topic_num, corpus_name, n=5, color="#667eea"):
    """특정 토픽의 대표 문서를 HTML로 반환"""

    topic_df = df[df['topic'] == topic_num].copy()

    if len(topic_df) == 0:
        return f"<p>⚠️ Topic {topic_num}에 해당하는 문서가 없습니다.</p>"

    top_docs = topic_df.nlargest(n, 'topic_probability')

    html = f"""
    <div class="topic-container">
        <div class="topic-header" style="background: linear-gradient(135deg, {color} 0%, {color}dd 100%);">
            📚 {corpus_name} - Topic {topic_num} (총 {len(topic_df)}건 중 Top {n})
        </div>
    """

    for i, (_, row) in enumerate(top_docs.iterrows(), 1):
        prob = row['topic_probability']
        title = row.get('title', '제목 없음')
        year = row.get('date', '')
        year_str = f" ({int(year)})" if pd.notna(year) else ""

        # 한국어 요약
        kor_sum = row.get('korean_summary', '')
        kor_html = ""
        if pd.notna(kor_sum) and str(kor_sum).strip():
            kor_text = str(kor_sum)[:400] + ('...' if len(str(kor_sum)) > 400 else '')
            kor_html = f"""
            <div class="section-label">🇰🇷 한국어 요약</div>
            <div class="section-content">{kor_text}</div>
            """

        # Abstract
        abstract = row.get('abstract', '')
        abs_html = ""
        if pd.notna(abstract) and str(abstract).strip():
            abs_text = str(abstract)[:400] + ('...' if len(str(abstract)) > 400 else '')
            abs_html = f"""
            <div class="section-label">📝 Abstract</div>
            <div class="section-content abstract-content">{abs_text}</div>
            """

        html += f"""
        <div class="doc-card">
            <div class="doc-header">
                <div>
                    <span class="doc-meta">#{i}{year_str}</span>
                    <div class="doc-title">{title}</div>
                </div>
                <span class="prob-badge">확률: {prob:.4f}</span>
            </div>
            <div class="doc-body">
                {kor_html}
                {abs_html}
            </div>
        </div>
        """

    html += "</div>"
    return html


def show_topic_docs(df, topic_num, corpus_name="", n=5):
    """특정 토픽의 대표 문서를 HTML로 표시"""
    color = "#667eea" if "academia" in corpus_name.lower() else "#e74c3c"
    html = STYLE + get_topic_docs_html(df, topic_num, corpus_name, n, color)
    display(HTML(html))


def compare_topics(acad_topic, ind_topic, n=5):
    """Academia와 Industry 토픽을 나란히 비교"""

    acad_html = get_topic_docs_html(df_academia, acad_topic, "ACADEMIA", n, "#667eea")
    ind_html = get_topic_docs_html(df_industry, ind_topic, "INDUSTRY", n, "#e74c3c")

    html = STYLE + f"""
    <h2 style="text-align:center; color:#2c3e50; margin:20px 0;">
        🔬 Topic 비교: Academia [{acad_topic}] vs Industry [{ind_topic}]
    </h2>
    <div class="comparison-wrapper">
        <div class="corpus-column">{acad_html}</div>
        <div class="corpus-column">{ind_html}</div>
    </div>
    """
    display(HTML(html))


def compare_topics_vertical(acad_topic, ind_topic, n=5):
    """Academia와 Industry 토픽을 세로로 비교 (좁은 화면용)"""

    acad_html = get_topic_docs_html(df_academia, acad_topic, "ACADEMIA", n, "#667eea")
    ind_html = get_topic_docs_html(df_industry, ind_topic, "INDUSTRY", n, "#e74c3c")

    html = STYLE + f"""
    <h2 style="text-align:center; color:#2c3e50; margin:20px 0;">
        🔬 Topic 비교: Academia [{acad_topic}] vs Industry [{ind_topic}]
    </h2>
    {acad_html}
    <div class="divider">⬇️ ⬇️ ⬇️</div>
    {ind_html}
    """
    display(HTML(html))


# ============================================================
# 4. 사용법
# ============================================================
"""
# 단일 토픽 확인
show_topic_docs(df_academia, 0, "Academia", n=5)
show_topic_docs(df_industry, 11, "Industry", n=5)

# 두 토픽 나란히 비교 (넓은 화면)
compare_topics(0, 11, n=5)

# 두 토픽 세로로 비교 (좁은 화면)
compare_topics_vertical(0, 11, n=5)
"""

# ============================================================
# 5. 바로 실행
# ============================================================
ACADEMIA_TOPIC = 3
INDUSTRY_TOPIC = 4
NUM_DOCS = 5

# 아래 중 하나 선택해서 실행
compare_topics(ACADEMIA_TOPIC, INDUSTRY_TOPIC, NUM_DOCS)  # 나란히
# compare_topics_vertical(ACADEMIA_TOPIC, INDUSTRY_TOPIC, NUM_DOCS)  # 세로로

Academia: 4800건 / Industry: 9665건


# Result - Academia Specific Topics 해석 만들기
- /home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv 문서에서 대표 문서 5건을 추려서 내가 무슨 내용을 담고 있는지 korean_summary, abstract이랑 같이 출력해주면 좋을 거 같아.
- 노트북 환경에서 바로 확인할 수 있게 만들어줘.

In [27]:
# ============================================================
# 🎓 Academia Specific Topics 뷰어 - Jupyter Notebook용
# 아래 코드를 노트북 셀에 복사하여 실행하세요
# ============================================================

import pandas as pd
from IPython.display import display, HTML

# 1. 데이터 로드
academia_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv"
df_academia = pd.read_csv(academia_path)

print(f"✅ Academia 데이터 로드 완료: {len(df_academia)}건")
print(f"📊 토픽 범위: {df_academia['topic'].min()} ~ {df_academia['topic'].max()}")

# 토픽별 문서 수 확인
topic_counts = df_academia['topic'].value_counts().sort_index()
print(f"\n📋 토픽별 문서 수:")
for t, c in topic_counts.items():
    print(f"   Topic {t}: {c}건")

# ============================================================
# 2. HTML 스타일 정의
# ============================================================

STYLE = """
<style>
.topic-container {
    font-family: 'Segoe UI', Arial, sans-serif;
    max-width: 100%;
    margin: 20px 0;
}
.topic-header {
    background: linear-gradient(135deg, #6B73FF 0%, #000DFF 100%);
    color: white;
    padding: 15px 20px;
    border-radius: 10px 10px 0 0;
    font-size: 18px;
    font-weight: bold;
}
.doc-card {
    border: 1px solid #e0e0e0;
    border-radius: 8px;
    margin: 10px 0;
    background: white;
    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}
.doc-header {
    background: #f8f9fa;
    padding: 12px 15px;
    border-bottom: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-between;
    align-items: center;
}
.doc-title {
    font-weight: bold;
    color: #2c3e50;
    font-size: 14px;
    flex: 1;
    margin-right: 10px;
}
.doc-meta {
    color: #7f8c8d;
    font-size: 12px;
}
.prob-badge {
    background: #6B73FF;
    color: white;
    padding: 4px 10px;
    border-radius: 15px;
    font-size: 12px;
    font-weight: bold;
    white-space: nowrap;
}
.doc-body {
    padding: 15px;
}
.section-label {
    font-weight: bold;
    color: #6B73FF;
    font-size: 12px;
    margin-bottom: 5px;
    display: flex;
    align-items: center;
    gap: 5px;
}
.section-content {
    color: #34495e;
    font-size: 13px;
    line-height: 1.7;
    margin-bottom: 12px;
    padding: 12px;
    background: #f8f9ff;
    border-radius: 5px;
    border-left: 3px solid #6B73FF;
}
.abstract-section .section-label {
    color: #3498db;
}
.abstract-section .section-content {
    background: #f8fcff;
    border-left-color: #3498db;
}
.topic-summary {
    background: #f0f0f0;
    padding: 10px 15px;
    border-radius: 0 0 10px 10px;
    font-size: 12px;
    color: #666;
}
</style>
"""

# ============================================================
# 3. 함수 정의
# ============================================================

def show_academia_topic(topic_num, n=5):
    """
    특정 Academia 토픽의 대표 문서를 HTML로 표시

    Parameters:
    - topic_num: 토픽 번호
    - n: 표시할 문서 수 (기본값: 5)
    """
    topic_df = df_academia[df_academia['topic'] == topic_num].copy()

    if len(topic_df) == 0:
        print(f"⚠️ Topic {topic_num}에 해당하는 문서가 없습니다.")
        return

    top_docs = topic_df.nlargest(n, 'topic_probability')

    html = STYLE + f"""
    <div class="topic-container">
        <div class="topic-header">
            🎓 Academia - Topic {topic_num} 대표 문서 Top {n}
        </div>
    """

    for i, (_, row) in enumerate(top_docs.iterrows(), 1):
        prob = row['topic_probability']
        title = row.get('title', '제목 없음')
        year = row.get('date', '')
        year_str = f"({int(year)})" if pd.notna(year) else ""

        # 한국어 요약
        kor_sum = row.get('korean_summary', '')
        kor_html = ""
        if pd.notna(kor_sum) and str(kor_sum).strip():
            kor_text = str(kor_sum)
            if len(kor_text) > 500:
                kor_text = kor_text[:500] + '...'
            kor_html = f"""
            <div class="section-label">🇰🇷 한국어 요약</div>
            <div class="section-content">{kor_text}</div>
            """

        # Abstract
        abstract = row.get('abstract', '')
        abs_html = ""
        if pd.notna(abstract) and str(abstract).strip():
            abs_text = str(abstract)
            if len(abs_text) > 500:
                abs_text = abs_text[:500] + '...'
            abs_html = f"""
            <div class="abstract-section">
                <div class="section-label">📝 Abstract</div>
                <div class="section-content">{abs_text}</div>
            </div>
            """

        html += f"""
        <div class="doc-card">
            <div class="doc-header">
                <div>
                    <span class="doc-meta">#{i} {year_str}</span>
                    <div class="doc-title">{title}</div>
                </div>
                <span class="prob-badge">확률: {prob:.4f}</span>
            </div>
            <div class="doc-body">
                {kor_html}
                {abs_html}
            </div>
        </div>
        """

    html += f"""
        <div class="topic-summary">
            📊 Topic {topic_num} 전체 문서 수: {len(topic_df)}건 | 평균 확률: {topic_df['topic_probability'].mean():.4f} | 최대 확률: {topic_df['topic_probability'].max():.4f}
        </div>
    </div>
    """

    display(HTML(html))


def show_all_topics(n=3):
    """
    모든 토픽의 대표 문서를 순차적으로 표시

    Parameters:
    - n: 각 토픽당 표시할 문서 수 (기본값: 3)
    """
    topics = sorted(df_academia['topic'].unique())
    print(f"📚 총 {len(topics)}개 토픽의 대표 문서 Top {n}씩 표시합니다.\n")

    for topic in topics:
        show_academia_topic(topic, n)


def topic_overview():
    """토픽별 요약 통계를 HTML 테이블로 표시"""
    summary = df_academia.groupby('topic').agg({
        'topic_probability': ['count', 'mean', 'max']
    }).round(4)
    summary.columns = ['문서 수', '평균 확률', '최대 확률']
    summary = summary.reset_index()

    html = STYLE + """
    <div class="topic-container">
        <div class="topic-header">📊 Academia 토픽 요약</div>
        <table style="width:100%; border-collapse: collapse; margin: 10px 0;">
            <tr style="background: #f0f0f0;">
                <th style="padding: 10px; border: 1px solid #ddd;">토픽</th>
                <th style="padding: 10px; border: 1px solid #ddd;">문서 수</th>
                <th style="padding: 10px; border: 1px solid #ddd;">평균 확률</th>
                <th style="padding: 10px; border: 1px solid #ddd;">최대 확률</th>
            </tr>
    """

    for _, row in summary.iterrows():
        html += f"""
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold;">Topic {int(row['topic'])}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">{int(row['문서 수'])}건</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">{row['평균 확률']:.4f}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">{row['최대 확률']:.4f}</td>
            </tr>
        """

    html += "</table></div>"
    display(HTML(html))


# ============================================================
# 4. 사용법
# ============================================================
"""
# 토픽 요약 보기
topic_overview()

# 특정 토픽 대표 문서 보기
show_academia_topic(0, n=5)   # Topic 0의 상위 5개 문서
show_academia_topic(3, n=5)   # Topic 3의 상위 5개 문서

# 모든 토픽 한번에 보기 (각 토픽당 3개씩)
show_all_topics(n=3)
"""

# ============================================================
# 5. 바로 실행 - 원하는 토픽 번호 수정 후 실행!
# ============================================================

TOPIC_NUM = 10    # 👈 확인하고 싶은 토픽 번호
NUM_DOCS = 5     # 👈 확인할 문서 수

# 실행 (주석 해제)
show_academia_topic(TOPIC_NUM, NUM_DOCS)

✅ Academia 데이터 로드 완료: 4800건
📊 토픽 범위: 0 ~ 11

📋 토픽별 문서 수:
   Topic 0: 442건
   Topic 1: 435건
   Topic 2: 801건
   Topic 3: 186건
   Topic 4: 273건
   Topic 5: 203건
   Topic 6: 26건
   Topic 7: 459건
   Topic 8: 202건
   Topic 9: 753건
   Topic 10: 454건
   Topic 11: 566건


# Result - Industry Specific Topics 해석 만들기
- /home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_industry_lda_bow.csv 문서에서 대표 문서 5건을 추려서 내가 무슨 내용을 담고 있는지 korean_summary, abstract이랑 같이 출력해주면 좋을 거 같아.
- 노트북 환경에서 바로 확인할 수 있게 만들어줘.

In [39]:
# ============================================================
# 🏢 Industry Specific Topics 뷰어 - Jupyter Notebook용
# 아래 코드를 노트북 셀에 복사하여 실행하세요
# ============================================================

import pandas as pd

# 1. 데이터 로드
industry_path = "/home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_industry_lda_bow.csv"
df_industry = pd.read_csv(industry_path)

print(f"✅ Industry 데이터 로드 완료: {len(df_industry)}건")
print(f"📊 토픽 범위: {df_industry['topic'].min()} ~ {df_industry['topic'].max()}")

# 토픽별 문서 수 확인
topic_counts = df_industry['topic'].value_counts().sort_index()
print(f"\n📋 토픽별 문서 수:")
# for t, c in topic_counts.items():
    # print(f"   Topic {t}: {c}건")

# ============================================================
# 2. 함수 정의
# ============================================================

def show_industry_topic(topic_num, n=5):
    """
    특정 Industry 토픽의 대표 문서 출력

    Parameters:
    - topic_num: 토픽 번호
    - n: 표시할 문서 수 (기본값: 5)
    """
    topic_df = df_industry[df_industry['topic'] == topic_num].copy()

    if len(topic_df) == 0:
        print(f"⚠️ Topic {topic_num}에 해당하는 문서가 없습니다.")
        return

    top_docs = topic_df.nlargest(n, 'topic_probability')

    print(f"\n{'='*80}")
    print(f"🏢 Industry - Topic {topic_num} 대표 문서 Top {n}")
    print(f"   (전체 {len(topic_df)}건 | 평균확률: {topic_df['topic_probability'].mean():.4f})")
    print(f"{'='*80}")

    for i, (_, row) in enumerate(top_docs.iterrows(), 1):
        prob = row['topic_probability']
        title = row.get('title', '제목 없음')
        year = row.get('date', '')
        year_str = f"({int(year)})" if pd.notna(year) else ""

        print(f"\n{'─'*80}")
        print(f"📄 [{i}] {year_str} 확률: {prob:.4f}")
        print(f"   📌 {title}")

        # 한국어 요약
        kor_sum = row.get('korean_summary', '')
        if pd.notna(kor_sum) and str(kor_sum).strip():
            kor_text = str(kor_sum)
            if len(kor_text) > 500:
                kor_text = kor_text[:500] + '...'
            print(f"\n   🇰🇷 한국어 요약:")
            print(f"      {kor_text}")

        # Abstract
        abstract = row.get('abstract', '')
        if pd.notna(abstract) and str(abstract).strip():
            abs_text = str(abstract)
            if len(abs_text) > 500:
                abs_text = abs_text[:500] + '...'
            print(f"\n   📝 Abstract:")
            print(f"      {abs_text}")

    print(f"\n{'='*80}\n")


def show_all_topics(n=3):
    """
    모든 토픽의 대표 문서를 순차적으로 표시

    Parameters:
    - n: 각 토픽당 표시할 문서 수 (기본값: 3)
    """
    topics = sorted(df_industry['topic'].unique())
    print(f"📚 총 {len(topics)}개 토픽의 대표 문서 Top {n}씩 표시합니다.\n")

    for topic in topics:
        show_industry_topic(topic, n)


def topic_overview():
    """토픽별 요약 통계 출력"""
    summary = df_industry.groupby('topic').agg({
        'topic_probability': ['count', 'mean', 'max']
    }).round(4)
    summary.columns = ['문서수', '평균확률', '최대확률']

    print(f"\n{'='*50}")
    print(f"📊 Industry 토픽 요약")
    print(f"{'='*50}")
    print(summary)
    print(f"{'='*50}\n")

    return summary


# ============================================================
# 3. 사용법
# ============================================================
"""
# 토픽 요약 보기
topic_overview()

# 특정 토픽 대표 문서 보기
show_industry_topic(0, n=5)   # Topic 0의 상위 5개 문서
show_industry_topic(11, n=5)  # Topic 11의 상위 5개 문서

# 모든 토픽 한번에 보기 (각 토픽당 3개씩)
show_all_topics(n=3)
"""

# ============================================================
# 4. 바로 실행 - 원하는 토픽 번호 수정 후 실행!
# ============================================================

TOPIC_NUM = 3    # 👈 확인하고 싶은 토픽 번호
NUM_DOCS = 5     # 👈 확인할 문서 수

# 실행 (주석 해제)
show_industry_topic(TOPIC_NUM, NUM_DOCS)

✅ Industry 데이터 로드 완료: 9665건
📊 토픽 범위: 0 ~ 15

📋 토픽별 문서 수:

🏢 Industry - Topic 3 대표 문서 Top 5
   (전체 650건 | 평균확률: 0.4964)

────────────────────────────────────────────────────────────────────────────────
📄 [1] (2025) 확률: 0.9545
   📌 OpenAI reverses course and says non-profit arm will retain control of firm

   📝 Abstract:
      OpenAIhas reversed course in the process of transforming into a for-profit entity, announcing on Monday that its non-profit arm would continue to control the business that makesChatGPTand other artificial intelligence (AI) products. Previously, the company had sought more independence for its for-profit division.
“We made the decision for the nonprofit to stay in control after hearing from civic leaders and having discussions with the offices of the Attorneys General of California and Delaware,”...

────────────────────────────────────────────────────────────────────────────────
📄 [2] (2025) 확률: 0.9398
   📌 Elon Musk loses bid to block OpenAI’s transformation.

   